# EDA: Handmade Traffic Law QA Data

Phân tích dữ liệu QA thủ công trong `data/processed/handmade`.

Mục tiêu:

- Kiểm tra dữ liệu có đủ trường bắt buộc không.
- Đánh giá độ dài câu hỏi, câu trả lời và đoạn căn cứ.
- Kiểm tra coverage theo văn bản, loại câu hỏi, chủ đề và căn cứ pháp lý.
- Tìm duplicate, near-duplicate và item có answer ít bám vào reference text.
- Tạo các biểu đồ cần thiết để đọc nhanh chất lượng dữ liệu.
- Tổng hợp đánh giá cuối cùng để quyết định dữ liệu dùng được ở mức nào.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import statistics
import tempfile
import unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "EDA":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

HANDMADE_DIR = PROJECT_ROOT / "data" / "processed" / "handmade"
print(f"Project root: {PROJECT_ROOT}")
print(f"Handmade data dir: {HANDMADE_DIR}")
print(f"Directory exists: {HANDMADE_DIR.exists()}")

In [ ]:
def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize("NFC", str(value or ""))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokens(value: Any) -> list[str]:
    return re.findall(r"[\w/.-]+", normalize_text(value), flags=re.UNICODE)


def token_count(value: Any) -> int:
    return len(tokens(value))


def quantiles(values: list[int | float]) -> dict[str, float]:
    values = sorted(v for v in values if v is not None)
    if not values:
        return {"min": 0, "p50": 0, "p90": 0, "p95": 0, "max": 0, "mean": 0}
    def q(p: float) -> float:
        idx = min(len(values) - 1, max(0, round((len(values) - 1) * p)))
        return float(values[idx])
    return {
        "min": float(values[0]),
        "p50": q(0.50),
        "p90": q(0.90),
        "p95": q(0.95),
        "max": float(values[-1]),
        "mean": float(statistics.mean(values)),
    }


def show_table(rows: Any, limit: int = 20):
    if pd is not None:
        if isinstance(rows, pd.DataFrame):
            display(rows.head(limit))
        else:
            display(pd.DataFrame(list(rows)).head(limit))
        return
    if isinstance(rows, list):
        for row in rows[:limit]:
            print(row)
    else:
        print(rows)


def jaccard(left: Any, right: Any) -> float:
    left_tokens = set(tokens(left))
    right_tokens = set(tokens(right))
    if not left_tokens and not right_tokens:
        return 1.0
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def answer_supported_by_reference(answer: Any, reference: Any, min_overlap: float = 0.25) -> bool:
    answer_terms = [term for term in tokens(answer) if len(term) > 2]
    if not answer_terms:
        return False
    reference_terms = set(tokens(reference))
    overlap = sum(1 for term in answer_terms if term in reference_terms) / len(answer_terms)
    return overlap >= min_overlap


def extract_reference_parts(value: Any) -> dict[str, str]:
    text = normalize_text(value)
    article_match = re.search(r"điều\s+(\d+[a-zA-Z]?)", text)
    clause_match = re.search(r"khoản\s+(\d+[a-zA-Z]?)", text)
    point_match = re.search(r"điểm\s+([a-zđ])", text)

    doc_type = "other"
    if "nghị định" in text:
        doc_type = "decree"
    elif "thông tư" in text or "qcvn" in text or "mục" in text:
        doc_type = "circular_or_standard"
    elif "luật" in text:
        doc_type = "law"

    return {
        "doc_type": doc_type,
        "article": article_match.group(1) if article_match else "",
        "clause": clause_match.group(1) if clause_match else "",
        "point": point_match.group(1) if point_match else "",
    }


def infer_question_type(question: Any, answer: Any, legal_basis: Any) -> str:
    text = normalize_text(f"{question} {answer} {legal_basis}")
    if any(term in text for term in ["mức phạt", "phạt tiền", "xử phạt", "trừ điểm"]):
        return "penalty"
    if any(term in text for term in ["biển", "báo hiệu", "vạch", "qcvn", "v85"]):
        return "sign_or_standard"
    if any(term in text for term in ["thủ tục", "hồ sơ", "cấp", "đổi", "thu hồi", "đào tạo", "sát hạch"]):
        return "procedure"
    if any(term in text for term in ["là gì", "được hiểu", "khái niệm", "bao gồm", "quy định về"]):
        return "definition"
    if any(term in text for term in ["điều kiện", "trường hợp", "khi nào", "phải làm gì", "không được"]):
        return "rule_or_condition"
    return "general"


def can_plot() -> bool:
    return pd is not None and plt is not None

## 2. Load Data

Dữ liệu handmade được đọc từ các file JSON trong `data/processed/handmade`. Mỗi item được bổ sung metadata để phục vụ phân tích.

In [ ]:
handmade_files = sorted(HANDMADE_DIR.glob("*.json"))
rows: list[dict[str, Any]] = []
file_errors: list[dict[str, Any]] = []

for path in handmade_files:
    try:
        items = read_json(path)
    except Exception as exc:
        file_errors.append({"source_file": path.name, "error": str(exc)})
        continue

    if not isinstance(items, list):
        file_errors.append({"source_file": path.name, "error": f"expected list, got {type(items).__name__}"})
        continue

    for index, item in enumerate(items):
        if not isinstance(item, dict):
            file_errors.append({"source_file": path.name, "row_index": index, "error": f"expected dict, got {type(item).__name__}"})
            continue

        reference_parts = extract_reference_parts(item.get("legal_basis"))
        row = {
            "source_file": path.name,
            "row_index": index,
            "question": item.get("question", ""),
            "answer": item.get("answer", ""),
            "legal_basis": item.get("legal_basis", ""),
            "reference_text": item.get("reference_text", ""),
            "question_norm": normalize_text(item.get("question")),
            "answer_norm": normalize_text(item.get("answer")),
            "legal_basis_norm": normalize_text(item.get("legal_basis")),
            "reference_text_norm": normalize_text(item.get("reference_text")),
            "question_tokens": token_count(item.get("question")),
            "answer_tokens": token_count(item.get("answer")),
            "reference_tokens": token_count(item.get("reference_text")),
            "answer_reference_jaccard": jaccard(item.get("answer"), item.get("reference_text")),
            "answer_supported_by_reference": answer_supported_by_reference(item.get("answer"), item.get("reference_text")),
            "question_type": infer_question_type(item.get("question"), item.get("answer"), item.get("legal_basis")),
            **reference_parts,
        }
        rows.append(row)

handmade_df = pd.DataFrame(rows) if pd is not None else None

print(f"Files: {len(handmade_files)}")
print(f"Rows: {len(rows)}")
print(f"File errors: {len(file_errors)}")
show_table(file_errors)
show_table(rows, limit=5)

## 3. Field Completeness

Các trường bắt buộc gồm `question`, `answer`, `legal_basis`, `reference_text`. Với bộ handmade dùng để kiểm tra RAG, các trường thiếu nên bằng 0.

In [ ]:
summary_by_file = []
for source_file in sorted({row["source_file"] for row in rows}):
    file_rows = [row for row in rows if row["source_file"] == source_file]
    summary_by_file.append({
        "source_file": source_file,
        "items": len(file_rows),
        "missing_question": sum(not row["question"] for row in file_rows),
        "missing_answer": sum(not row["answer"] for row in file_rows),
        "missing_legal_basis": sum(not row["legal_basis"] for row in file_rows),
        "missing_reference_text": sum(not row["reference_text"] for row in file_rows),
        "median_question_tokens": quantiles([row["question_tokens"] for row in file_rows])["p50"],
        "median_answer_tokens": quantiles([row["answer_tokens"] for row in file_rows])["p50"],
        "median_reference_tokens": quantiles([row["reference_tokens"] for row in file_rows])["p50"],
    })

show_table(summary_by_file, limit=50)

## 4. Length Analysis

Độ dài câu hỏi, câu trả lời và reference text cho biết dữ liệu có đủ ngữ cảnh nhưng không quá dài hay không.

In [ ]:
length_summary = [
    {"field": "question_tokens", **quantiles([row["question_tokens"] for row in rows])},
    {"field": "answer_tokens", **quantiles([row["answer_tokens"] for row in rows])},
    {"field": "reference_tokens", **quantiles([row["reference_tokens"] for row in rows])},
]
length_issues = [
    {"issue": "question_lt_5_tokens", "count": sum(row["question_tokens"] < 5 for row in rows)},
    {"issue": "answer_lt_5_tokens", "count": sum(row["answer_tokens"] < 5 for row in rows)},
    {"issue": "reference_lt_5_tokens", "count": sum(row["reference_tokens"] < 5 for row in rows)},
    {"issue": "answer_gt_120_tokens", "count": sum(row["answer_tokens"] > 120 for row in rows)},
    {"issue": "reference_gt_200_tokens", "count": sum(row["reference_tokens"] > 200 for row in rows)},
]
show_table(length_summary)
show_table(length_issues)

## 5. Legal Basis Coverage

Phân tích `legal_basis` theo loại văn bản, điều, khoản, điểm. Đây là cơ sở để đánh giá citation granularity.

In [ ]:
legal_basis_summary = [
    {"metric": "rows", "value": len(rows)},
    {"metric": "doc_types", "value": dict(Counter(row["doc_type"] for row in rows).most_common())},
    {"metric": "with_article", "value": sum(bool(row["article"]) for row in rows)},
    {"metric": "with_clause", "value": sum(bool(row["clause"]) for row in rows)},
    {"metric": "with_point", "value": sum(bool(row["point"]) for row in rows)},
    {"metric": "missing_parseable_article", "value": sum(not row["article"] for row in rows)},
]
show_table(legal_basis_summary)

basis_by_file = []
for source_file in sorted({row["source_file"] for row in rows}):
    file_rows = [row for row in rows if row["source_file"] == source_file]
    basis_by_file.append({
        "source_file": source_file,
        "items": len(file_rows),
        "doc_types": dict(Counter(row["doc_type"] for row in file_rows).most_common()),
        "with_article": sum(bool(row["article"]) for row in file_rows),
        "with_clause": sum(bool(row["clause"]) for row in file_rows),
        "with_point": sum(bool(row["point"]) for row in file_rows),
    })
show_table(basis_by_file, limit=50)

## 6. Question Type And Topic Coverage

Question type được suy luận bằng rule đơn giản. Topic coverage giúp biết bộ handmade đang phủ tốt chủ đề nào và thiếu chủ đề nào.

In [ ]:
question_type_counts = Counter(row["question_type"] for row in rows)
show_table([{"question_type": key, "count": value} for key, value in question_type_counts.most_common()])

question_type_by_file = []
for source_file in sorted({row["source_file"] for row in rows}):
    file_rows = [row for row in rows if row["source_file"] == source_file]
    question_type_by_file.append({"source_file": source_file, **dict(Counter(row["question_type"] for row in file_rows))})
show_table(question_type_by_file, limit=50)

In [ ]:
topic_patterns = {
    "penalty": ["phạt", "xử phạt", "trừ điểm", "mức phạt"],
    "license": ["giấy phép lái xe", "gplx", "hạng", "sát hạch", "đào tạo"],
    "traffic_sign": ["biển", "báo hiệu", "vạch", "qcvn"],
    "vehicle": ["xe", "phương tiện", "ô tô", "mô tô", "xe máy"],
    "road_rule": ["dừng", "đỗ", "chuyển làn", "tốc độ", "vượt", "nhường đường"],
    "procedure": ["thủ tục", "hồ sơ", "cấp", "đổi", "thu hồi", "cấp lại"],
    "accident_or_control": ["tai nạn", "tuần tra", "kiểm soát", "cảnh sát", "hiệu lệnh"],
}

topic_rows = []
for topic, keywords in topic_patterns.items():
    count = 0
    file_counts = Counter()
    for row in rows:
        text = normalize_text(f"{row['question']} {row['answer']} {row['legal_basis']} {row['reference_text']}")
        if any(keyword in text for keyword in keywords):
            count += 1
            file_counts[row["source_file"]] += 1
    topic_rows.append({"topic": topic, "count": count, "files": dict(file_counts.most_common())})

show_table(sorted(topic_rows, key=lambda item: item["count"], reverse=True))

## 7. Answer Support From Reference Text

Metric kiểm tra câu trả lời có bám vào `reference_text` không. Đây là kiểm tra rule-based, dùng để ưu tiên review thủ công.

In [ ]:
support_summary = [
    {"metric": "answer_supported_by_reference", "value": sum(row["answer_supported_by_reference"] for row in rows)},
    {"metric": "answer_not_supported_by_reference", "value": sum(not row["answer_supported_by_reference"] for row in rows)},
    {"metric": "median_answer_reference_jaccard", "value": quantiles([row["answer_reference_jaccard"] for row in rows])["p50"]},
    {"metric": "low_overlap_count_lt_0_15", "value": sum(row["answer_reference_jaccard"] < 0.15 for row in rows)},
]
show_table(support_summary)

low_support_rows = sorted(
    [row for row in rows if not row["answer_supported_by_reference"] or row["answer_reference_jaccard"] < 0.15],
    key=lambda row: row["answer_reference_jaccard"],
)
show_table([
    {
        "source_file": row["source_file"],
        "row_index": row["row_index"],
        "question": row["question"][:140],
        "legal_basis": row["legal_basis"],
        "jaccard": round(row["answer_reference_jaccard"], 3),
        "answer_preview": row["answer"][:160],
        "reference_preview": row["reference_text"][:160],
    }
    for row in low_support_rows[:20]
], limit=20)

## 8. Duplicate, Near-Duplicate And Leakage Checks

Kiểm tra duplicate, near-duplicate và leakage mô phỏng train/test bằng similarity câu hỏi.

In [ ]:
question_counts = Counter(row["question_norm"] for row in rows)
answer_counts = Counter(row["answer_norm"] for row in rows)
exact_duplicate_questions = [question for question, count in question_counts.items() if question and count > 1]
exact_duplicate_answers = [answer for answer, count in answer_counts.items() if answer and count > 1]

duplicate_summary = [
    {"metric": "exact_duplicate_questions", "value": len(exact_duplicate_questions)},
    {"metric": "exact_duplicate_answers", "value": len(exact_duplicate_answers)},
]
show_table(duplicate_summary)

near_duplicates = []
for left_index, left in enumerate(rows):
    for right in rows[left_index + 1:]:
        score = SequenceMatcher(None, left["question_norm"], right["question_norm"]).ratio()
        if score >= 0.88:
            near_duplicates.append({
                "score": round(score, 3),
                "left_file": left["source_file"],
                "left_index": left["row_index"],
                "right_file": right["source_file"],
                "right_index": right["row_index"],
                "left_question": left["question"][:160],
                "right_question": right["question"][:160],
            })
near_duplicates = sorted(near_duplicates, key=lambda item: item["score"], reverse=True)
print(f"Near duplicate question pairs >= 0.88: {len(near_duplicates)}")
show_table(near_duplicates, limit=20)

In [ ]:
# Deterministic split simulation without external dependencies.
sorted_rows = sorted(rows, key=lambda row: (row["source_file"], row["row_index"]))
test_rows = [row for index, row in enumerate(sorted_rows) if index % 5 == 0]
train_rows = [row for index, row in enumerate(sorted_rows) if index % 5 != 0]

leakage_candidates = []
for test_row in test_rows:
    best_score = 0.0
    best_train = None
    for train_row in train_rows:
        score = SequenceMatcher(None, test_row["question_norm"], train_row["question_norm"]).ratio()
        if score > best_score:
            best_score = score
            best_train = train_row
    if best_score >= 0.88 and best_train is not None:
        leakage_candidates.append({
            "score": round(best_score, 3),
            "test_file": test_row["source_file"],
            "test_index": test_row["row_index"],
            "train_file": best_train["source_file"],
            "train_index": best_train["row_index"],
            "test_question": test_row["question"][:160],
            "nearest_train_question": best_train["question"][:160],
        })

print(f"Simulated test rows: {len(test_rows)}")
print(f"Leakage candidates >= 0.88: {len(leakage_candidates)}")
show_table(sorted(leakage_candidates, key=lambda item: item["score"], reverse=True), limit=20)

## 9. Lexical Diversity

Type-token ratio đo độ đa dạng từ vựng. Giá trị thấp thường xảy ra khi dữ liệu cùng domain và có nhiều mẫu câu lặp.

In [ ]:
def type_token_ratio(values: list[str]) -> float:
    all_tokens = []
    for value in values:
        all_tokens.extend(tokens(value))
    if not all_tokens:
        return 0.0
    return len(set(all_tokens)) / len(all_tokens)

lexical_summary = [
    {"field": "question", "ttr": type_token_ratio([row["question"] for row in rows])},
    {"field": "answer", "ttr": type_token_ratio([row["answer"] for row in rows])},
    {"field": "reference_text", "ttr": type_token_ratio([row["reference_text"] for row in rows])},
]
show_table(lexical_summary)

term_counts = Counter()
for row in rows:
    term_counts.update(term for term in tokens(row["question"]) if len(term) > 2)
show_table([{"term": term, "count": count} for term, count in term_counts.most_common(30)], limit=30)

## 10. Optional NLP Checks

Các kiểm tra POS/NER chỉ chạy nếu `underthesea` có trong môi trường. Kết quả chỉ dùng để tham khảo vì NER tổng quát không tối ưu cho pháp lý giao thông.

In [ ]:
try:
    from underthesea import ner, pos_tag
except ImportError:
    ner = None
    pos_tag = None

if ner is None or pos_tag is None:
    print("underthesea is not installed; skipping POS/NER checks.")
else:
    pos_counts = Counter()
    ner_counts = Counter()
    entity_counts = Counter()
    for row in rows:
        for _, tag in pos_tag(row["question_norm"]):
            pos_counts[tag] += 1
        for word, _, _, label in ner(row["question_norm"]):
            if label != "O":
                ner_counts[label] += 1
                entity_counts[f"{word} ({label})"] += 1
    show_table([{"pos_tag": tag, "count": count} for tag, count in pos_counts.most_common(20)])
    show_table([{"ner_label": label, "count": count} for label, count in ner_counts.most_common(20)])
    show_table([{"entity": entity, "count": count} for entity, count in entity_counts.most_common(20)])

## 11. Visualizations

Biểu đồ giúp đọc nhanh phân bố dữ liệu, coverage và các lỗi cần ưu tiên review.

In [ ]:
if can_plot():
    df = pd.DataFrame(rows)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    df["source_file"].value_counts().sort_values().plot(kind="barh", ax=axes[0, 0], color="#4C78A8")
    axes[0, 0].set_title("QA Items By File")
    axes[0, 0].set_xlabel("Items")

    df["question_type"].value_counts().sort_values().plot(kind="barh", ax=axes[0, 1], color="#54A24B")
    axes[0, 1].set_title("Question Type Distribution")
    axes[0, 1].set_xlabel("Items")

    axes[1, 0].hist(df["question_tokens"], bins=30, alpha=0.8, label="question", color="#4C78A8")
    axes[1, 0].hist(df["answer_tokens"], bins=30, alpha=0.6, label="answer", color="#F58518")
    axes[1, 0].set_title("Question And Answer Length")
    axes[1, 0].set_xlabel("Tokens")
    axes[1, 0].set_ylabel("Items")
    axes[1, 0].legend()

    df["doc_type"].value_counts().sort_values().plot(kind="barh", ax=axes[1, 1], color="#B279A2")
    axes[1, 1].set_title("Legal Basis Document Type")
    axes[1, 1].set_xlabel("Items")

    plt.tight_layout()
    plt.show()
else:
    print("Install pandas and matplotlib to render charts.")

In [ ]:
if can_plot():
    topic_df = pd.DataFrame(topic_rows).sort_values("count", ascending=True)
    issue_counts = pd.Series({
        "missing_question": sum(not row["question"] for row in rows),
        "missing_answer": sum(not row["answer"] for row in rows),
        "missing_legal_basis": sum(not row["legal_basis"] for row in rows),
        "missing_reference_text": sum(not row["reference_text"] for row in rows),
        "exact_duplicate_questions": len(exact_duplicate_questions),
        "near_duplicate_pairs": len(near_duplicates),
        "low_answer_reference_overlap": sum(row["answer_reference_jaccard"] < 0.15 for row in rows),
        "leakage_candidates": len(leakage_candidates),
    }).sort_values(ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    topic_df.plot(kind="barh", x="topic", y="count", legend=False, ax=axes[0], color="#72B7B2")
    axes[0].set_title("Topic Coverage")
    axes[0].set_xlabel("Items")

    issue_counts.plot(kind="barh", ax=axes[1], color="#E45756")
    axes[1].set_title("Main Data Issues")
    axes[1].set_xlabel("Count")

    plt.tight_layout()
    plt.show()
else:
    print("Install pandas and matplotlib to render charts.")

## 12. Final Handmade Data Assessment

Bộ handmade QA hiện có **556 items** từ **7 file** trong `data/processed/handmade`:

- `168-nd-cp.signed.json`: 50 items
- `336nd.signed.json`: 50 items
- `35-2024-qh15.json`: 106 items
- `35-bgtvt.json`: 100 items
- `36-2024-qh15.json`: 50 items
- `36-2024-qh15_tiep.json`: 100 items
- `51-bgtvt-kem.json`: 100 items

Không có lỗi đọc file. Các trường bắt buộc `question`, `answer`, `legal_basis`, `reference_text` đều có dữ liệu, nên tập này đủ điều kiện làm bộ kiểm tra thủ công/regression cho RAG.

### Data Shape

Độ dài dữ liệu đang ở mức hợp lý cho legal QA:

- `question_tokens`: median 18, p95 25, max 32.
- `answer_tokens`: median 18, p95 40, max 84.
- `reference_tokens`: median 38, p95 85, max 165.

Điểm cần chú ý:

- Có **1 question dưới 5 tokens**.
- Có **31 answers dưới 5 tokens**.
- Không có reference text dưới 5 tokens.
- Không có answer trên 120 tokens hoặc reference trên 200 tokens.

Kết luận: câu hỏi và reference nhìn chung đủ ngữ cảnh. Một số answer rất ngắn nên cần review vì có thể chưa đủ thông tin để đánh giá chất lượng trả lời.

### Legal Basis Quality

Căn cứ pháp lý parse được ở mức tốt cho luật/nghị định/thông tư:

- `with_article`: 456 / 556
- `with_clause`: 448 / 556
- `with_point`: 224 / 556
- `missing_parseable_article`: 100 / 556

Phần thiếu article chủ yếu đến từ `51-bgtvt-kem.json`, vì QCVN/tiêu chuẩn kỹ thuật dùng `Mục`, bảng, loại đường, biển báo thay vì mẫu `Điều/Khoản/Điểm`. Đây không phải lỗi nghiêm trọng, nhưng evaluator cần xử lý citation cho QCVN khác với văn bản luật thông thường.

### Coverage

Phân bố loại câu hỏi hiện tại:

- `general`: 141
- `procedure`: 123
- `sign_or_standard`: 109
- `penalty`: 100
- `rule_or_condition`: 42
- `definition`: 41

Tập handmade đang phủ được các route chính của hệ thống: xử phạt, thủ tục/GPLX, biển báo/QCVN, quy tắc giao thông và định nghĩa. Tuy nhiên `definition` và `rule_or_condition` ít hơn các nhóm còn lại, nên nếu dùng làm benchmark chính thức cần cân nhắc tăng mẫu cho hai nhóm này.

### Answer Grounding

Kết quả kiểm tra answer bám vào reference:

- `answer_supported_by_reference`: 533 / 556
- `answer_not_supported_by_reference`: 23 / 556
- `median_answer_reference_jaccard`: 0.4909
- `low_overlap_count_lt_0_15`: 67 / 556

Đây là tín hiệu tốt ở mức tổng thể, nhưng **67 item overlap thấp** cần review. Một số trường hợp có thể vẫn đúng vì answer diễn giải lại reference, nhưng cũng có thể là reference text thiếu hoặc answer đi quá xa căn cứ.

### Duplication And Leakage Risk

Kết quả kiểm tra trùng lặp:

- Exact duplicate questions: 0
- Exact duplicate answers: 22
- Near-duplicate question pairs >= 0.88: 16
- Simulated leakage candidates >= 0.88: 6 / 112 test rows

Không có duplicate question tuyệt đối là tốt. Tuy nhiên near-duplicate và leakage candidates cho thấy không nên split random theo row nếu dùng tập này làm benchmark. Nên split theo `source_file`, `legal_basis`, hoặc nhóm điều luật để tránh cùng một nội dung xuất hiện ở cả dev/test.

### Lexical Diversity

Type-token ratio:

- Question TTR: 0.0942
- Answer TTR: 0.1120
- Reference TTR: 0.0640

TTR thấp là bình thường với dữ liệu pháp lý cùng domain, vì nhiều cụm như `đường`, `người`, `giấy phép`, `biển`, `phạt`, `điều` lặp lại nhiều. Điều này củng cố nhu cầu dùng split theo căn cứ pháp lý thay vì random, vì random split dễ tạo leakage từ vựng và cấu trúc câu.

### Final Decision

Bộ `data/processed/handmade` hiện **đủ tốt để dùng làm smoke test, regression test và kiểm tra thủ công chất lượng RAG**. Nó có đủ câu hỏi, đáp án, căn cứ pháp lý và đoạn reference để kiểm tra answer/citation.

Tuy nhiên, để dùng như benchmark chính thức, cần xử lý trước:

1. Review 67 item có `answer_reference_jaccard < 0.15`.
2. Review 23 item `answer_supported_by_reference=False`.
3. Review 16 cặp near-duplicate question.
4. Thiết kế split theo `source_file` hoặc `legal_basis`, không split random theo row.
5. Với QCVN/biển báo, chuẩn hóa citation theo `Mục/Bảng/Biển` thay vì ép về `Điều/Khoản/Điểm`.

Kết luận kỹ thuật: dữ liệu handmade đang đạt mức **usable for controlled RAG validation**, nhưng chưa đạt mức **final benchmark** nếu chưa review grounding, near-duplicate và split leakage.